Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/COAD_P_PDC000117"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                    Gene     Sequence                                     Glycan                            
ENSP00000262776@541                     LGALS3BP AAIPSALDTNSSK                                N2H9F0S0G0            16.142057   
ENSP00000273784@166;ENSP00000393887@165 AHSG     AALAAFNAQNNGSNFQLEEISR                       N4H5F0S1G0            16.632591   
                                                                                              N4H5F0S2G0            17.440971   
                                                                                              N4H5F1S1G0            14.590123   
                                                                                              N4H5F1S2G0            16.015287   
...                                                                                                                       ...   
ENSP00000376793@267;ENSP00000451119@49  SERPINA3 YTGNASALFILPDQDKMEEVEAMLLPETLK               N7H6F0S2G0            12.328709   
ENSP00000261590@458                     DSG2     YVQNGTYTVK                                   N5H5F3S0G0            12.578579   
                                                                                              N6H6F2S1G0            10.609755   
ENSP00000502432@874                     MUC2     YYDFDGHCSYVAVQDYCGQNSSLGSFSIITENVPCGTTGVTCSK N4H5F1S0G0            12.600595   
                                                                                              N6H8F0S3G0            10.205955   

                                                                                                          11CO037_N_01  \
Site                                    Gene     Sequence                                     Glycan                     
ENSP00000262776@541                     LGALS3BP AAIPSALDTNSSK                                N2H9F0S0G0     15.752013   
ENSP00000273784@166;ENSP00000393887@165 AHSG     AALAAFNAQNNGSNFQLEEISR                       N4H5F0S1G0     16.640067   
                                                                                              N4H5F0S2G0     17.906996   
                                                                                              N4H5F1S1G0     13.936824   
                                                                                              N4H5F1S2G0     16.887327   
...                                                                                                                ...   
ENSP00000376793@267;ENSP00000451119@49  SERPINA3 YTGNASALFILPDQDKMEEVEAMLLPETLK               N7H6F0S2G0           NaN   
ENSP00000261590@458                     DSG2     YVQNGTYTVK                                   N5H5F3S0G0           NaN   
                                                                                              N6H6F2S1G0           NaN   
ENSP00000502432@874                     MUC2     YYDFDGHCSYVAVQDYCGQNSSLGSFSIITENVPCGTTGVTCSK N4H5F1S0G0           NaN   
                                                                                              N6H8F0S3G0           NaN   

                                                                                                          11CO051_N_01  \
Site                                    Gene     Sequence                                     Glycan                     
ENSP00000262776@541                     LGALS3BP AAIPSALDTNSSK                                N2H9F0S0G0     16.448905   
ENSP00000273784@166;ENSP00000393887@165 AHSG     AALAAFNAQNNGSNFQLEEISR                       N4H5F0S1G0     16.795787   
                                                                                              N4H5F0S2G0     18.002834   
                                                                                              N4H5F1S1G0     13.139796   
                                                                                              N4H5F1S2G0     

In [4]:
meta_path= os.path.join(meta_dir, "COAD_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,...,TCF7L2_mutation,ZFP36L2_mutation,ANO10_mutation,MXRA8_mutation,B2M_mutation,CTNNB1_mutation,UPF3A_mutation,ZNF540_mutation,BMPR2_mutation,PTX4_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,...,BIN,BIN,BIN,BIN,BIN,BIN,BIN,BIN,BIN,BIN
0,01CO001,60.0,Male,NaN,NaN,NaN,pT4,pN2,Stage III,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01CO005,69.0,Female,NaN,NaN,NaN,pT3,pN0,Stage II,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,01CO006,75.0,Female,NaN,NaN,NaN,pT4,pN2,Stage III,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,01CO008,54.0,Female,NaN,NaN,NaN,pT3,pN0,Stage II,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,01CO013,57.0,Male,NaN,NaN,NaN,pT2,pN0,Stage I,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,24CO005,41.0,Female,NaN,NaN,NaN,pT2,pN1,Stage III,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,27CO004,50.0,Female,NaN,NaN,NaN,pT3,pN2,Stage III,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
106,05CO004,40.0,Female,NaN,NaN,NaN,pT3,pN1,Stage III,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,01CO001,Male,Stage III
1,01CO005,Female,Stage II
2,01CO006,Female,Stage III
3,01CO008,Female,Stage II
4,01CO013,Male,Stage I
...,...,...,...
104,24CO005,Female,Stage III
105,27CO004,Female,Stage III
106,05CO004,Female,Stage III
107,13CO001,Male,NaN


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,01CO001,Male,Stage III
1,01CO005,Female,Stage II
2,01CO006,Female,Stage III
3,01CO008,Female,Stage II
4,01CO013,Male,Stage I
5,01CO014,Female,Stage III
6,01CO015,Male,Stage I
7,01CO019,Female,Stage III
8,01CO022,Female,Stage I
9,05CO002,Female,Stage III


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

97

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
3,01CO008_T_01,Female,Stage II
103,22CO006_T_01,Female,Stage III
9,05CO002_T_01,Female,Stage III
62,11CO036_T_01,Male,Stage III
98,20CO006_T_01,Male,Stage IV
7,01CO019_T_01,Female,Stage III
38,09CO005_T_01,Female,Stage III
33,05CO053_T_02,Male,Stage II
28,05CO045_T_02,Female,Stage III
23,05CO035_T_02,Female,Stage II


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

,,,,Intensity.Reference,11CO037_N_01,11CO051_N_01,01CO008_T_01,22CO006_T_01,05CO002_T_01,11CO036_T_01,20CO006_T_01,01CO019_T_01,09CO005_T_01,...,05CO011_T_22,09CO018_N_22,11CO079_N_22,01CO022_N_22,09CO014_N_22,05CO047_N_22,06CO001_N_22,16CO003_N_22,ref_22,pool_22
Site,Gene,Sequence,Glycan,,,,,,,,,,,,,,,,,,,,,
ENSP00000262776@541,LGALS3BP,AAIPSALDTNSSK,N2H9F0S0G0,16.142057,15.752013,16.448905,16.930430,15.948791,16.667042,16.912199,16.949683,16.721377,17.094491,...,16.081493,16.141128,15.549939,15.626056,15.729305,15.198984,15.656971,15.586182,15.993241,16.142057
ENSP00000273784@166;ENSP00000393887@165,AHSG,AALAAFNAQNNGSNFQLEEISR,N4H5F0S1G0,16.632591,16.640067,16.795787,16.716761,16.617350,17.068240,16.272056,16.837435,16.885535,17.061946,...,16.798411,16.652695,16.411245,16.842951,16.455673,16.506527,16.909670,16.608174,16.506500,16.632591


In [14]:
data_df.shape

(21107, 221)

In [15]:
samples = meta3['Sample.ID'].to_list()

In [16]:
len(samples)

97

In [17]:
df2 = data_df.loc[:,samples].dropna()

In [18]:
df2.shape

(112, 97)

In [19]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [20]:
data2.shape

(83, 97)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [21]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [22]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [23]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
1,ENSP00000511601@925;ENSP00000480150@925;ENSP00...,HM
2,ENSP00000264613@345@CP@AGLQAFFQVQECNK@N4H5F0S2G0,only_S
3,ENSP00000265983@446@HPX@ALPQPQNVTSLLGCTH@N6H3F...,only_F
4,ENSP00000410815@859;ENSP00000418996@708@ENSG00...,F+S
...,...,...
78,ENSP00000350406@271;ENSP00000348170@235;ENSP00...,only_S
79,ENSP00000413723@183;ENSP00000052754@292;ENSP00...,only_F
80,ENSP00000413723@183;ENSP00000052754@292;ENSP00...,F+S
81,ENSP00000376802@267;ENSP00000386094@267@SERPIN...,only_S


Map glycan types with colors.

In [24]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [25]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [26]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)